# Task 2: DenseNet-121

This notebook loads Notebook 1's prepared arrays and trains only DenseNet-121. Shared neural-network
augmentation, batching, checkpointing, recovery, and evaluation live in `src/task2_utils.py`.


## How to Run

Run `01_task2_setup.ipynb` first. Change the settings below if needed, then run this notebook.
It does not rerun setup or another model.


## 1. Setup


In [ ]:
%matplotlib inline
import torch.nn as nn
from IPython.display import display


In [ ]:
from pathlib import Path
import sys

REPO_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                  if (p / "pyproject.toml").exists()), None)
if REPO_ROOT is None:
    raise FileNotFoundError("Could not find the repository root containing pyproject.toml")
sys.path.insert(0, str(REPO_ROOT))

from torchvision.models import densenet121
from src.task2_utils import (
    NeuralTrainer, REPO_ROOT, ensure_task2_directories, export_model_results,
    neural_training_config, prepared_namespace, result_frame,
)
ensure_task2_directories()


### 1.1 Training configuration


In [ ]:
QUICK_RUN = True
RANDOM_STATE = 42
RESUME = False
ALLOW_CPU = False

TRAINING_CONFIG = neural_training_config(
    quick_run=QUICK_RUN, random_state=RANDOM_STATE, resume=RESUME, allow_cpu=ALLOW_CPU,
)


## 2. Load Prepared Data and Shared Training Pipeline


In [ ]:
data = prepared_namespace()
trainer = NeuralTrainer(data, TRAINING_CONFIG)
print(f"Loaded {len(data.train_frame):,} train and {len(data.validation_frame):,} validation rows")
print(f"Training batches per epoch: {len(trainer.train_loader)}")


## 3. Train DenseNet-121


In [ ]:
def build_model():
    model = densenet121(weights=None)
    model.classifier = nn.Linear(model.classifier.in_features, data.n_classes)
    return model

densenet, history, logits = trainer.train_or_restore(
    "densenet121", build_model, "DenseNet-121"
)
predictions = logits.argmax(axis=1)
display(result_frame(data.y_val, predictions, logits, "DenseNet-121"))
trainer.plot_history(history, "DenseNet-121 training")


## 4. Export DenseNet-121 Results


In [ ]:
output_dir = export_model_results(
    "densenet121", "DenseNet-121", data, logits, history, scores_are_logits=True,
)
print("Saved model:", REPO_ROOT / "models" / "task2_densenet121.pt")
print("Saved outputs:", output_dir)
